[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/36_int8_quantization.ipynb)

# 🔴 Hard: INT8 Quantized Linear

Implement a **post-training quantized linear layer** using INT8 weights.

### Signature
```python
class Int8Linear(nn.Module):
    def __init__(self, weight: Tensor, bias: Tensor = None): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Quantization (per-channel)
1. `scale = weight.abs().max(dim=1) / 127`
2. `weight_int8 = round(weight / scale).clamp(-128, 127).to(int8)`
3. Store as `register_buffer` (not trainable)
4. Forward: dequantize (`int8.float() * scale`) then matmul

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

In [15]:
class Int8Linear(nn.Module):
    def __init__(self, weight, bias=None):
        """
        weight: (out_feat, in_feat)
        """
        super().__init__()
        #once I find the absmax I can transform all the others: X_old : X_new = absmax : 127 --> X_new = X_old * 127 / absmax --> X_new = X_old / scale with scale = absmax / 127
        self.scale = weight.abs().max(dim = 1, keepdim = True).values / 127 #(out_feat,)
        weight_int8 = torch.round(weight / self.scale).to(torch.int8)

        self.register_buffer("weight_int8", weight_int8) #useful since it will be excluded from model.parameters(), NOT TRAINABLE, callable through "name" from outside and self. from inside


    def forward(self, x):
        weight = self.weight_int8.float() * self.scale
        return x @ weight.T

In [16]:
# 🧪 Debug
w = torch.randn(8, 4)
q = Int8Linear(w)
x = torch.randn(2, 4)
print('Output:', q(x).shape)
print('dtype:', q.weight_int8.dtype)
print('Max quant error:', (w - q.weight_int8.float() * q.scale).abs().max().item())

Output: torch.Size([2, 8])
dtype: torch.int8
Max quant error: 0.007197916507720947


In [17]:
# ✅ SUBMIT
from torch_judge import check
check('int8_quantization')


🧪 Testing: INT8 Quantized Linear (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Weight is int8 (0.7ms)
  ✅ [2/5] Values in [-128, 127] (2.3ms)
  ✅ [3/5] Dequantized close to original (12.3ms)
  ✅ [4/5] Forward output shape (0.4ms)
  ✅ [5/5] Weight is buffer not parameter (0.3ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (16.0ms total)
  Progress saved. Run status() to see your dashboard.

